# Prueba de la API — Predicción de cancelaciones hoteleras

Notebook para consumir la API desplegada en Render.  
Con la URL que nos proporciona Render ya podemos consumir la API desde cualquier sitio. Sustituye `BASE_URL` por la URL real de tu servicio.

In [1]:
import requests

BASE_URL = 'https://tc-m-03-despliegue-modelo-hu9q.onrender.com'

---
## Documentación interactiva
FastAPI genera automáticamente una documentación interactiva accesible: `BASE_URL\docs`   
Para nuestra API: https://tc-m-03-despliegue-modelo-hu9q.onrender.com/docs  
  
Desde esta interfaz es posible:
- ver todos los endpoints disponibles
- entender los parámetros requeridos
- probar la API directamente desde el navegador

---
## Endpoint 1 - Landing page `/`
Muestra todos los endpoints disponibles y cómo usarlos.

In [2]:
response = requests.get(BASE_URL + '/')
print(response.status_code)
print(response.text)

200
{"mensaje":"API de predicción de cancelaciones hoteleras","documentacion":"/docs"}


---
## Endpoint 2 — Predicción `/predict`
Los parámetros se pasan como **query string** (`params=...`).  
El modelo devuelve la predicción (0/1), la etiqueta y las probabilidades de cada clase.

| Parámetro | Tipo | Valores |
|---|---|---|
| hotel | str | Resort Hotel \| City Hotel |
| customer_type | str | Transient \| Contract \| Group \| Transient-Party |
| market_segment | str | Online TA \| Offline TA/TO \| Direct \| Corporate \| ... |
| deposit_type | str | No Deposit \| Non Refund \| Refundable |
| meal | str | BB \| HB \| FB \| SC \| Undefined |
| country | str | Código ISO-3166 (PRT, ESP, GBR, USA...) |
| distribution_channel | str | TA/TO \| Direct \| Corporate \| GDS \| Undefined |
| reserved_room_type | str | A \| B \| C \| D \| E \| F \| G \| H \| L |
| is_repeated_guest | int | 0 = cliente nuevo \| 1 = repetido |
| lead_time | int | días de antelación |
| previous_cancellations | int | cancelaciones previas |
| adults | int | número de adultos |
| days_in_waiting_list | int | días en lista de espera |
| adr | float | precio medio diario |
| previous_bookings_not_canceled | int | reservas previas no canceladas |
| booking_changes | int | cambios en la reserva |
| required_car_parking_spaces | int | plazas de parking |
| total_of_special_requests | int | peticiones especiales |

In [5]:
# Reserva 1: perfil de BAJO riesgo de cancelación
# Cliente repetido, sin cancelaciones previas, poca antelación, depósito no reembolsable
params = {
    'hotel': 'Resort Hotel',
    'customer_type': 'Transient',
    'market_segment': 'Direct',
    'deposit_type': 'Non Refund',
    'meal': 'BB',
    'country': 'PRT',
    'distribution_channel': 'Direct',
    'reserved_room_type': 'A',
    'is_repeated_guest': 1,
    'lead_time': 10,
    'previous_cancellations': 0,
    'adults': 2,
    'days_in_waiting_list': 0,
    'adr': 90,
    'previous_bookings_not_canceled': 5,
    'booking_changes': 0,
    'required_car_parking_spaces': 1,
    'total_of_special_requests': 2,
}
response = requests.post(BASE_URL + '/predict', json=params)
print(response.status_code)
print(response.json())

200
{'prediction': 0, 'prediction_label': 'No cancelada', 'probability_cancelation': 0.366}


In [9]:
# Reserva 2: perfil de ALTO riesgo de cancelación
# Sin depósito, mucha antelación, historial de cancelaciones, sin peticiones especiales
params = {
    'hotel': 'City Hotel',
    'customer_type': 'Transient',
    'market_segment': 'Online TA',
    'deposit_type': 'No Deposit',
    'meal': 'SC',
    'country': 'GBR',
    'distribution_channel': 'TA/TO',
    'reserved_room_type': 'D',
    'is_repeated_guest': 0,
    'lead_time':  300,
    'previous_cancellations': 3,
    'adults':  2,
    'days_in_waiting_list': 0,
    'adr': 65,
    'previous_bookings_not_canceled': 0,
    'booking_changes': 0,
    'required_car_parking_spaces': 0,
    'total_of_special_requests': 0,
}
response = requests.post(BASE_URL + '/predict', json=params)
print(response.status_code)
print(response.json())

200
{'prediction': 0, 'prediction_label': 'No cancelada', 'probability_cancelation': 0.2429}


In [12]:
# Reserva 3: con valores faltantes — la API imputa y avisa
params = {
    'hotel':            'City Hotel',
    'customer_type':    'Transient',
    'market_segment':   'Online TA',
    'deposit_type':     'No Deposit',
    'meal':             'BB',
    'country':          'ESP',
}
response = requests.post(BASE_URL + '/predict', json=params)
print(response.status_code)
print(response.json())

422
{'detail': [{'type': 'missing', 'loc': ['body', 'distribution_channel'], 'msg': 'Field required', 'input': {'hotel': 'City Hotel', 'customer_type': 'Transient', 'market_segment': 'Online TA', 'deposit_type': 'No Deposit', 'meal': 'BB', 'country': 'ESP'}}, {'type': 'missing', 'loc': ['body', 'reserved_room_type'], 'msg': 'Field required', 'input': {'hotel': 'City Hotel', 'customer_type': 'Transient', 'market_segment': 'Online TA', 'deposit_type': 'No Deposit', 'meal': 'BB', 'country': 'ESP'}}, {'type': 'missing', 'loc': ['body', 'is_repeated_guest'], 'msg': 'Field required', 'input': {'hotel': 'City Hotel', 'customer_type': 'Transient', 'market_segment': 'Online TA', 'deposit_type': 'No Deposit', 'meal': 'BB', 'country': 'ESP'}}, {'type': 'missing', 'loc': ['body', 'lead_time'], 'msg': 'Field required', 'input': {'hotel': 'City Hotel', 'customer_type': 'Transient', 'market_segment': 'Online TA', 'deposit_type': 'No Deposit', 'meal': 'BB', 'country': 'ESP'}}, {'type': 'missing', 'loc

---
## Endpoint 2 — Info del modelo `/health`

> **Nota para la exposición:** este endpoint está comentado en `app.py`.  
> Para activarlo en vivo durante la demo:  
> 1. Descomenta las líneas en `app.py`  
> 2. En bash: `git add app.py && git commit -m "add health endpoint" && git push origin main`  
> 3. Render redespliega automáticamente (~1-2 min)  
> 4. Ejecuta la celda de abajo

In [ ]:
# Ejecutar tras el redespliegue en la exposición
response = requests.get(BASE_URL + '/health')
print(response.status_code)
print(response.json())

200
{'mensaje': 'Nuevo endpoint desplegado correctamente', 'status': 'ok'}


### **¡Enhorabuena!** Has desplegado tu API de manera remota en Render :)